# Estimating daily evapotranspiration in coastal wetlands — and mapping it at 30 m

**I-GUIDE Summer School 2026 · Team 2**

## Problem statement
Coastal wetlands are among the most productive, carbon-rich ecosystems on Earth, and how
much water they return to the atmosphere — their **evapotranspiration (ET)** — governs
their water and carbon balance. ET is measured directly only at a handful of
**eddy-covariance flux towers**, each sensing a footprint of a few hundred metres. This
notebook asks: *can we learn daily ET from satellite + weather predictors well enough to
map it where there is no tower?* — and then produces 30 m ET maps over unmonitored
National Estuarine Research Reserve (NERR) wetlands.

## Datasets
- **Target:** energy-balance-closed daily ET at **13 U.S. coastal-wetland flux towers**
  (AmeriFlux/FLUXNET, 2018–2023), spanning the Everglades, the Gulf and Atlantic coasts,
  and the Sacramento–San Joaquin Delta.
- **Predictors (sampled over a 500 m window at each tower on cloud-free overpass days):**
  seven **Landsat** indices (NDVI, SAVI, EVI2, NDWI, MNDWI, LAI, land-surface temperature)
  and seven **meteorological** variables (air temperature, VPD, shortwave radiation, wind
  speed, gridMET FAO-56 reference ET, and a cyclic day-of-year).
- **Analysis table:** `data/processed/more_sites_table.parquet` (833 records).
- **Prediction targets:** seven NERR reserve boundaries in `shp_predict/`.

## Data availability
The analysis table and the 30 m ET maps are published as a Knowledge Element on the
**I-GUIDE Platform**:
<https://platform.i-guide.io/datasets/a0a5736a-4a53-4fb5-b20d-33d4e6019992>
(direct download:
`https://storage.i-guide.io/datasets/a0a5736a-4a53-4fb5-b20d-33d4e6019992/coastal_et_dataset.zip`).
The setup cell below fetches the training table from there automatically if it isn't already
present locally, so this notebook runs even on its own.

## What this notebook does (Run All)
1. **Part 1** — describe the data. **Part 2** — compare 12 ML models under 3 cross-validation
schemes and select features four independent ways. **Part 3** — validate spatial upscaling.
**Part 4** — map ET at 30 m over the seven reserves.

**Runs in ~10 minutes with no raw-data download** (most of that is the 12-model comparison in
Part 2; the maps load in seconds). Everything needed is included: the
analysis table drives Parts 1–3, and Part 4's 30 m maps are **pre-computed**
(`data/processed/reserve_maps/`) and loaded by default. The full raw→features pipeline is
*already baked into* the analysis table, so you do **not** need the ~20 GB of raw imagery.
To regenerate the reserve maps from scratch instead (re-download Landsat + gridMET, ~10 min),
set `RECOMPUTE = True` in Part 4. (Part 1 makes one small Landsat *catalog* query — metadata
only — to illustrate scene availability; that needs internet, as does `RECOMPUTE`.) The
bootstrap cell auto-installs dependencies and, if the table is missing, fetches it from the
published dataset — so this notebook runs as-is on the **I-GUIDE JupyterHub**. Paths
self-resolve; nothing to edit.

---

In [ ]:
# ---- bootstrap: locate the project, install dependencies, fetch data (runs anywhere) ----
import os, sys, subprocess, importlib.util

# 1) project root: env override, else search cwd / parent / grandparent for data/processed
#    (robust whether this notebook is opened from repo-root or from notebooks/ on any host)
ROOT = os.environ.get("COASTAL_ET_ROOT")
if not ROOT or not os.path.isdir(os.path.join(ROOT, "data", "processed")):
    here = os.getcwd()
    cands = [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
    ROOT = next((c for c in cands if os.path.isdir(os.path.join(c, "data", "processed"))),
                "/anvil/projects/x-ees260113/team2/coastal-et")
PROC = f"{ROOT}/data/processed"; FIG = f"{ROOT}/figures"; os.makedirs(FIG, exist_ok=True)
print("project root:", ROOT)

# 2) ensure the geospatial + ML stack is importable; if not, pip install requirements.txt
_missing = [m for m in ["geopandas","rasterio","rioxarray","stackstac","pystac_client",
                        "planetary_computer","xgboost","lightgbm","statsmodels"]
            if importlib.util.find_spec(m) is None]
if _missing:
    req = next((p for p in [f"{ROOT}/requirements.txt", "requirements.txt", "../requirements.txt"]
                if os.path.exists(p)), None)
    print("installing dependencies", _missing, "from", req or "PyPI")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] +
                   (["-r", req] if req else _missing), check=False)

# 3) fetch the published analysis table if absent, so the notebook runs even on its own
if not os.path.exists(f"{PROC}/more_sites_table.parquet"):
    import io, zipfile, urllib.request
    os.makedirs(PROC, exist_ok=True)
    url = "https://storage.i-guide.io/datasets/a0a5736a-4a53-4fb5-b20d-33d4e6019992/coastal_et_dataset.zip"
    print("fetching published dataset from I-GUIDE ...")
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    with z.open("coastal_et_dataset/more_sites_table.parquet") as s, \
         open(f"{PROC}/more_sites_table.parquet", "wb") as d:
        d.write(s.read())
    print("saved analysis table")

In [ ]:
# ---- imports, plotting config, and the portable per-reserve pipeline (used in Part 4) ----
import glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, f"{ROOT}/src")
import reserve_et as RE

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8, "axes.linewidth": 0.7, "figure.dpi": 120,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7})
INK = "#1a1a1a"
print("shapefiles :", RE.SHP_DIR)

---
# Part 1 — The data: sites, features, and how ET is measured

## Problem statement

Coastal wetlands are among the most productive and carbon-rich ecosystems on Earth, and
**evapotranspiration (ET)** — the water they return to the atmosphere — is central to their
water and energy balance. ET is measured directly only at a handful of **eddy-covariance
flux towers**, each representing a tiny footprint, so we cannot observe it across the vast,
inaccessible wetland landscape. **Our problem: can we learn the relationship between ET and
satellite + meteorological predictors at the towers, and use it to map daily ET at 30 m over
*unmonitored* coastal wetlands?** This matters for blue-carbon accounting, water-resource and
restoration management, and validating satellite ET products in a setting where they are
rarely tested. The core scientific question is whether such a model **transfers to a new,
unseen site** — which we test explicitly with leave-site-out cross-validation.

## 1. Study sites

We load the site metadata and keep the sites that made it into the modelling table
(a cloud-free overpass matched to a measured-ET day). These 13 span the Everglades plus
Atlantic/Gulf/Pacific coastal wetlands — the diversity that makes upscaling work.

In [ ]:
meta = pd.read_csv(f"{PROC}/core_coastal_sites.csv")
table = pd.read_parquet(f"{PROC}/more_sites_table.parquet")
model_sites = sorted(table.SITE_ID.unique())
inv = meta[meta.SITE_ID.isin(model_sites)][
    ["SITE_ID","SITE_NAME","STATE","IGBP","LAT","LON","KOEPPEN","MAT","MAP"]].copy()
# attach sample count + mean ET from the table
agg = table.groupby("SITE_ID").agg(n_overpass=("ET_closed_mm","size"),
                                    mean_ET=("ET_closed_mm","mean")).round(2)
inv = inv.merge(agg, on="SITE_ID").sort_values("LAT")
print(f"{len(inv)} modelling sites across {inv.STATE.nunique()} states")
inv

### Where they are

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(inv.LON, inv.LAT, c=inv.mean_ET, s=90, cmap="YlGnBu",
                edgecolor="#333", linewidth=0.6, zorder=3, vmin=2, vmax=5)
for _, r in inv.iterrows():
    ax.annotate(r.SITE_ID, (r.LON, r.LAT), xytext=(4, 4), textcoords="offset points", fontsize=6.5)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Coastal-wetland flux towers (colour = mean daily ET)", fontsize=10, fontweight="bold")
cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02); cb.set_label("mean ET (mm/day)")
ax.grid(alpha=0.25, zorder=0)
plt.show()

## 2. Downloading the data (live methods)

We don't re-download everything here (the full extraction is a cluster job), but the
cells below run the **actual access methods** so you can see how each stream is pulled.

**2a. Flux towers — AmeriFlux/FLUXNET.** We use the ONEFlux FLUXNET product (half-hourly
LE, H, Rn, G, plus ERA5 met). Each site's page and metadata:

In [ ]:
print("example AmeriFlux site pages:")
for _, r in inv.head(4).iterrows():
    print(f"  {r.SITE_ID}  {r.SITE_NAME:<40} https://ameriflux.lbl.gov/sites/siteinfo/{r.SITE_ID}")

**2b. Satellite — Planetary Computer STAC.** A live query for clear Landsat scenes over
one site (this hits the network; needs a compute-backed session).

In [ ]:
try:
    import planetary_computer as pc, pystac_client
    site = inv.iloc[len(inv)//2]
    cat = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                    modifier=pc.sign_inplace)
    items = list(cat.search(collections=["landsat-c2-l2"],
                 bbox=[site.LON-0.05, site.LAT-0.05, site.LON+0.05, site.LAT+0.05],
                 datetime="2022-01-01/2022-12-31",
                 query={"eo:cloud_cover": {"lt": 10}}).items())
    print(f"{site.SITE_ID}: {len(items)} clear Landsat scenes in 2022")
    if items:
        it = sorted(items, key=lambda i: i.properties['eo:cloud_cover'])[0]
        print("  clearest:", it.id, f"{it.properties['eo:cloud_cover']:.1f}% cloud")
        print("  assets used: red, nir08, green, swir16, lwir11 (thermal), qa_pixel")
except Exception as _e:
    print('  [skipped — live data service unavailable:', type(_e).__name__, _e, ']')

**2c. Meteorology — gridMET via OPeNDAP.** A live point fetch (no big download): a few
days of reference ET and temperature at one site.

In [ ]:
try:
    import xarray as xr
    GM = "http://thredds.northwestknowledge.net:8080/thredds/dodsC/agg_met_{v}_1979_CurrentYear_CONUS.nc"
    eto = xr.open_dataset(GM.format(v="pet"))["daily_mean_reference_evapotranspiration_grass"]
    pt = eto.sel(lat=site.LAT, lon=site.LON, method="nearest").sel(
         day=slice("2022-06-01", "2022-06-07"))
    print(f"gridMET ETo at {site.SITE_ID}, first week of June 2022 (mm/day):")
    print(np.round(pt.values, 2))
except Exception as _e:
    print('  [skipped — live data service unavailable:', type(_e).__name__, _e, ']')

## 3. Preprocessing

**3a. Energy-balance closure → ET.** Eddy covariance under-closes the surface energy
balance (H+LE < Rn−G). We correct LE (Bowen-ratio / ONEFlux `LE_CORR`), convert to ET
with a temperature-dependent latent heat, and aggregate to daily (≥80% coverage). The
correction is not cosmetic — it moves ET by 10–30%, so we keep both.

In [ ]:
et = pd.read_parquet(f"{PROC}/daily_closed_et.parquet")
clo = (et[et.ET_closed_mm.notna()].groupby("SITE_ID")
       .agg(ET_open=("ET_open_mm","mean"), ET_closed=("ET_closed_mm","mean"),
            closure=("CLOSURE","first")).round(2))
clo["uplift_%"] = ((clo.ET_closed/clo.ET_open - 1)*100).round(1)
clo = clo[clo.index.isin(model_sites)].sort_values("uplift_%")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
y = np.arange(len(clo))
ax.barh(y, clo["uplift_%"], color="#4C72B0", height=0.7, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(clo.index, fontsize=7)
ax.set_xlabel("ET increase from energy-balance closure (%)")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
ax.set_title("Closure correction raises ET by 10–30%", fontsize=10, fontweight="bold")
ax.grid(axis="x", alpha=0.25, zorder=0)
plt.show()
clo

**3b. Satellite indices — computed per pixel, then averaged.** Because indices are
nonlinear, we compute them on each pixel before window-averaging (ratio-of-means is biased
over mixed marsh/water). The exact formulas we use:

```
NDVI  = (NIR − Red) / (NIR + Red)
SAVI  = 1.5·(NIR − Red) / (NIR + Red + 0.5)
EVI2  = 2.5·(NIR − Red) / (NIR + 2.4·Red + 1)
NDWI  = (NIR − SWIR) / (NIR + SWIR)
MNDWI = (Green − SWIR) / (Green + SWIR)
LAI   = −ln((0.69 − SAVI) / 0.59) / 0.91     (DisALEXI/TSEB relation)
LST   = Landsat thermal band (K)             (Sentinel-2 has no thermal)
```
These are the 7 satellite features. LST is Landsat-only, which is why Landsat is
indispensable and Sentinel-2 only densifies the optical record.

**3c. Flux-footprint weighting (Kljun et al. 2015).** The tower sees a
footprint-integrated flux, so we weight satellite pixels by their 2-D footprint
contribution (Obukhov length from u*/H/T, σv≈1.9u*, literature tower heights). This is a
heavy precompute (`src/footprint_climatology.py`); the modelling table already carries the
footprint-weighted features.

## 4. Data overview (the modelling table)

Everything below is computed live from `more_sites_table.parquet` — the analysis-ready
matches of footprint-weighted satellite + meteorology to measured daily ET.

In [ ]:
SAT = ["LAI","EVI2","SAVI","NDVI","NDWI","MNDWI","LST_K"]
MET = ["TA_ERA","VPD_ERA","SW_IN_ERA","WS_ERA","ETo_mm","DOY_sin","DOY_cos"]
print(f"{len(table)} overpass matches | {table.SITE_ID.nunique()} sites | "
      f"years {int(table.year.min())}–{int(table.year.max())}")
print(f"target: ET_closed_mm  (mean {table.ET_closed_mm.mean():.2f}, "
      f"range {table.ET_closed_mm.min():.2f}–{table.ET_closed_mm.max():.2f} mm/day)")
table[["SITE_ID","year"] + SAT + MET + ["ET_closed_mm"]].describe().round(2).T[["mean","std","min","max"]]

### ET distribution by site

In [ ]:
order = table.groupby("SITE_ID").ET_closed_mm.median().sort_values().index
data = [table[table.SITE_ID == s].ET_closed_mm.values for s in order]
fig, ax = plt.subplots(figsize=(7.4, 3.8))
bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.6,
                medianprops=dict(color="#222"), flierprops=dict(ms=2, alpha=0.3))
for p in bp["boxes"]:
    p.set(facecolor="#6BCC5C", edgecolor="#333", linewidth=0.6)
ax.set_xticks(range(1, len(order)+1)); ax.set_xticklabels(order, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("daily ET (mm/day)")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
ax.set_title("Measured daily ET by site", fontsize=10, fontweight="bold")
plt.show()

### Overpass coverage — matches per site per year

In [ ]:
piv = table.pivot_table(index="SITE_ID", columns="year", values="ET_closed_mm",
                        aggfunc="size", fill_value=0)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
im = ax.imshow(piv.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns.astype(int))
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index, fontsize=7)
for i in range(len(piv.index)):
    for j in range(len(piv.columns)):
        v = piv.values[i, j]
        if v: ax.text(j, i, int(v), ha="center", va="center", fontsize=6,
                      color="white" if v > piv.values.max()*0.5 else "#333")
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("matches")
ax.set_title("Cloud-free overpass matches per site-year", fontsize=10, fontweight="bold")
plt.show()

### Feature distributions

In [ ]:
feats = SAT[:6] + ["LST_K","TA_ERA","VPD_ERA","SW_IN_ERA","WS_ERA","ETo_mm"]
fig, axes = plt.subplots(3, 4, figsize=(9, 6))
for ax, f in zip(axes.ravel(), feats):
    ax.hist(table[f].dropna(), bins=30, color="#4C72B0", alpha=0.85)
    ax.set_title(f, fontsize=8); ax.tick_params(labelsize=6)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
for ax in axes.ravel()[len(feats):]: ax.axis("off")
fig.suptitle("Predictor distributions across all site-days", fontsize=11, fontweight="bold")
fig.tight_layout()
plt.show()

### How features relate to ET

In [ ]:
cols = SAT + ["TA_ERA","VPD_ERA","SW_IN_ERA","WS_ERA","ETo_mm","ET_closed_mm"]
corr = table[cols].corr()["ET_closed_mm"].drop("ET_closed_mm").sort_values()
fig, ax = plt.subplots(figsize=(5.2, 4.2))
col = ["#55A868" if f in SAT else "#4C72B0" for f in corr.index]
ax.barh(np.arange(len(corr)), corr.values, color=col, height=0.7, zorder=3)
ax.set_yticks(np.arange(len(corr))); ax.set_yticklabels(corr.index, fontsize=7.5)
ax.axvline(0, color="#888", lw=0.8)
ax.set_xlabel("Pearson r with daily ET")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc="#55A868", label="satellite"), Patch(fc="#4C72B0", label="meteorology")],
          frameon=False, fontsize=7, loc="lower right")
ax.set_title("Correlation of each predictor with ET", fontsize=10, fontweight="bold")
plt.show()

### Takeaways
- **13 diverse coastal wetlands**, ~830 cloud-free overpass matches, 2022–2023 focus.
- **Energy-balance closure** raises ET 10–30% — larger than most model differences.
- Meteorology (**ETo**, VPD, SW_in) correlates most strongly with ET; among satellite
  features the **water/moisture indices and LST** lead, raw greenness is weaker — the
  same signal the model's feature importance shows later.

Next: **`02_model_selection.ipynb`** trains models on exactly this table.

## Data sources, licensing & citations

All inputs are open data; our derived products are released **CC-BY-4.0**.

- **AmeriFlux / FLUXNET (ONEFlux)** — flux ET (target). CC-BY-4.0. Pastorello et al. (2020),
  *Scientific Data* 7:225; cite each site's data DOI.
- **Landsat Collection-2 L2** — reflectance + surface temperature. USGS, public domain.
- **Sentinel-2 L2A** — optical reflectance. Copernicus/ESA, free & open.
- **Microsoft Planetary Computer** — STAC access to the imagery.
- **gridMET** — reference ET & meteorology. Abatzoglou (2013), *Int. J. Climatol.* 33:121–131.
- **ERA5** — meteorology. Hersbach et al. (2020), *QJRMS* 146:1999–2049 (Copernicus C3S).
- **Kljun et al. (2015)** — flux-footprint model (robustness test). *GMD* 8:3695–3713.
- **scikit-learn** — Pedregosa et al. (2011), *JMLR* 12:2825–2830 (BSD-3).

Full list and the dataset description are in `docs/CITATIONS.md` and
`data/processed/DATASET_README.md`.

**How to run this project:** see `00_environment_setup.ipynb` — register the shared
`Python (coastal-et)` kernel, then run `01 → 02 → 03 → 04`. Notebooks 02 and 03 are fully
offline (just the processed table); 01 and 04 fetch imagery/meteorology live.

---
# Part 2 — Model comparison and cross-validation

## Load the data

One self-contained table: 833 overpass matches across 13 coastal-wetland towers.

In [ ]:
# The 14 predictors: 7 satellite + 7 meteorology. Target is measured closed ET.
SAT = ["LAI", "EVI2", "SAVI", "NDVI", "NDWI", "MNDWI", "LST_K"]
MET = ["TA_ERA", "VPD_ERA", "SW_IN_ERA", "WS_ERA", "ETo_mm", "DOY_sin", "DOY_cos"]
FEATS = SAT + MET
EVERGLADES = ["US-Esm", "US-TaS", "US-Skr", "US-Elm", "US-EvM"]

d = pd.read_parquet(f"{PROC}/more_sites_table.parquet")
SITES = sorted(d.SITE_ID.unique())
print(f"{len(d)} overpass matches | {len(SITES)} sites | {len(FEATS)} features")
print("target: ET_closed_mm (measured, closure-corrected daily ET)")
d[["SITE_ID", "year"] + FEATS + ["ET_closed_mm"]].head()

## Cross-validation, honestly

The evaluator refits the model for every fold/year/site so nothing leaks across the
split we care about.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error

def evaluate(make_model, data, scheme, feats=FEATS):
    """Return (R2, MAE, y_true, y_pred) for one CV scheme.
    kfold=predict at monitored sites; year=predict unseen years;
    site=predict a completely unseen tower (spatial upscaling)."""
    yt, yp = [], []
    if scheme == "kfold":
        for tri, tei in KFold(10, shuffle=True, random_state=0).split(data):
            m = clone(make_model()).fit(data.iloc[tri][feats].values, data.iloc[tri].ET_closed_mm.values)
            yp.append(m.predict(data.iloc[tei][feats].values)); yt.append(data.iloc[tei].ET_closed_mm.values)
    elif scheme == "year":
        for s in sorted(data.SITE_ID.unique()):
            ds = data[data.SITE_ID == s]
            for y in sorted(ds.year.dropna().unique()):
                tr, te = ds[ds.year != y], ds[ds.year == y]
                if len(te) < 5 or len(tr) < 15: continue
                m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
                yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    else:  # leave-site-out
        for s in sorted(data.SITE_ID.unique()):
            tr, te = data[data.SITE_ID != s], data[data.SITE_ID == s]
            if len(te) < 5: continue
            m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
            yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    return r2_score(yt, yp), mean_absolute_error(yt, yp), yt, yp

print("CV evaluator ready (schemes: kfold, year, site)")

## The model zoo

Linear, kernel, tree-ensemble, Gaussian-process and (if installed) boosted-tree models.

In [ ]:
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def model_zoo():
    z = {
        "Ridge":       lambda: make_pipeline(StandardScaler(), Ridge(alpha=10)),
        "ElasticNet":  lambda: make_pipeline(StandardScaler(), ElasticNet(alpha=0.05, l1_ratio=0.3, max_iter=5000)),
        "PLS":         lambda: make_pipeline(StandardScaler(), PLSRegression(n_components=6)),
        "kNN":         lambda: make_pipeline(StandardScaler(), KNeighborsRegressor(10, weights="distance")),
        "SVR":         lambda: make_pipeline(StandardScaler(), SVR(C=5, gamma="scale", epsilon=0.2)),
        "GaussProc":   lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
                            kernel=ConstantKernel() * RBF() + WhiteKernel(), normalize_y=True, alpha=1e-3, random_state=0)),
        "RandomForest": lambda: RandomForestRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
        "ExtraTrees":  lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
        "GradBoost":   lambda: GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, random_state=0),
        "HistGBM":     lambda: HistGradientBoostingRegressor(max_iter=400, l2_regularization=1, random_state=0),
    }
    try:
        from xgboost import XGBRegressor
        z["XGBoost"] = lambda: XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.03,
                                            subsample=0.8, colsample_bytree=0.8, random_state=0, verbosity=0)
    except Exception: pass
    try:
        from lightgbm import LGBMRegressor
        z["LightGBM"] = lambda: LGBMRegressor(n_estimators=400, num_leaves=31, learning_rate=0.03,
                                              subsample=0.8, random_state=0, verbose=-1)
    except Exception: pass
    return z

ZOO = model_zoo()
print("model zoo:", list(ZOO.keys()))

## Train everything

This is the actual training run — each model is fit across all three schemes.

In [ ]:
# Train every model across all three schemes. This actually fits the models now
# (~1-3 min). Leave-site-out is the headline test: can we predict an UNSEEN tower?
rows = []
for name, mk in ZOO.items():
    rk, _, _, _ = evaluate(mk, d, "kfold")
    ry, _, _, _ = evaluate(mk, d, "year")
    rs, _, _, _ = evaluate(mk, d, "site")
    rows.append((name, rk, ry, rs))
    print(f"  {name:<13} kfold={rk:5.2f}  leave-year={ry:5.2f}  leave-site={rs:5.2f}", flush=True)

comp = pd.DataFrame(rows, columns=["model", "kfold", "leave_year", "leave_site"]).sort_values(
    "leave_site", ascending=False).reset_index(drop=True)
print(f"\nbest upscaler (leave-site): {comp.iloc[0].model}  R2={comp.iloc[0].leave_site:.2f}")
comp

## Visualize the comparison

In [ ]:
# Figure: R2 by validation scheme for the top models (draw it here, save a copy)
top = comp.head(6)
x = np.arange(len(top)); w = 0.26
fig, ax = plt.subplots(figsize=(6.4, 3.6), constrained_layout=True)
for i, (col, lab, c) in enumerate([("kfold", "K-fold (monitored)", "#B0C4DE"),
                                   ("leave_year", "leave-year (unseen years)", "#6b9bc3"),
                                   ("leave_site", "leave-site (unseen tower)", "#2C5F8A")]):
    ax.bar(x + (i-1)*w, np.clip(top[col], -0.2, 1), w, label=lab, color=c, zorder=3)
ax.axhline(0, color="#8a8a8a", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(top.model, rotation=30, ha="right", fontsize=7.5)
ax.set_ylabel("$R^2$"); ax.set_ylim(-0.2, 1.0)
ax.legend(frameon=False, fontsize=7, loc="upper right")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Predictive skill by validation scheme (13 sites)", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/model_comparison_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

We consistently find **tree ensembles (ExtraTrees / RandomForest)** give the best
leave-site skill, while the **Gaussian process** wins K-fold interpolation. Deep nets are
omitted here — at n≈833 they don't beat the trees and add instability.

## Feature importance — what actually transfers

Gini importance flatters whatever the trees split on in-sample. The honest measure is
permutation importance on **held-out sites**: shuffle a feature, see how much unseen-tower
skill drops.

In [ ]:
# Feature importance the honest way: permutation importance measured on HELD-OUT
# sites (leave-site-out), averaged over sites. This ranks what actually transfers.
from sklearn.inspection import permutation_importance
perm = np.zeros(len(FEATS)); n = 0
for s in SITES:
    tr, te = d[d.SITE_ID != s], d[d.SITE_ID == s]
    if len(te) < 5: continue
    m = ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1).fit(
        tr[FEATS].values, tr.ET_closed_mm.values)
    pi = permutation_importance(m, te[FEATS].values, te.ET_closed_mm.values, n_repeats=8, random_state=0)
    perm += np.clip(pi.importances_mean, 0, None); n += 1
imp = pd.Series(perm / n, index=FEATS).sort_values()

col = ["#55A868" if f in SAT else "#4C72B0" for f in imp.index]
fig, ax = plt.subplots(figsize=(4.6, 4.2), constrained_layout=True)
ax.barh(np.arange(len(imp)), imp.values, color=col, height=0.72, zorder=3)
ax.set_yticks(np.arange(len(imp))); ax.set_yticklabels(imp.index, fontsize=7.5)
ax.set_xlabel("Permutation importance (leave-site-out $\\Delta R^2$)")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc="#55A868", label="satellite"), Patch(fc="#4C72B0", label="meteorology")],
          frameon=False, fontsize=7, loc="lower right")
ax.set_title("What drives transferable ET skill", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/feature_importance_runnable.png", dpi=200, bbox_inches="tight")
plt.show()
print("Top drivers:", list(imp.sort_values(ascending=False).index[:5]))

Reference ET (`ETo_mm`) dominates, followed by seasonal timing and the water/moisture
indices (NDWI, LST); raw greenness (LAI/NDVI/EVI2/SAVI) barely moves unseen-site skill —
the marshes are too spectrally similar in greenness for it to discriminate.

## 8. Feature-group ablation — how few / which inputs do we need?

Before any formal selection, we just train on progressively fewer (or different) input
groups and score each on leave-site-out. This shows directly how much each group adds.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
GROUPS = {
    "ETo only (1)":         ["ETo_mm"],
    "meteorology (7)":      MET,
    "greenness (4)":        ["LAI", "EVI2", "SAVI", "NDVI"],
    "water + LST (3)":      ["NDWI", "MNDWI", "LST_K"],
    "satellite (7)":        SAT,
    "met + water/LST (10)": MET + ["NDWI", "MNDWI", "LST_K"],
    "FULL (14)":            FEATS,
}
# the top-k most important inputs (ranking from the section-7 permutation importance)
ranked = imp.sort_values(ascending=False).index.tolist()
GROUPS["top-6 (importance)"] = ranked[:6]
GROUPS["top-7 (importance)"] = ranked[:7]
print("top-7 by importance:", ranked[:7])
etm = lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1)
rows = []
for name, cols in GROUPS.items():
    rs = evaluate(etm, d, "site", feats=cols)[0]
    rows.append((name, len(cols), round(rs, 3)))
    print(f"  {name:<22} n={len(cols):<3} leave-site R2 = {rs:.3f}", flush=True)
abl = pd.DataFrame(rows, columns=["feature set", "n", "leave_site_R2"])
abl

In [ ]:
order = abl.sort_values("leave_site_R2")
col = ["#C44E52" if s in ("greenness (4)", "ETo only (1)") else
       ("#DD8452" if ("satellite" in s or "water" in s) else "#4C72B0") for s in order["feature set"]]
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.barh(np.arange(len(order)), order.leave_site_R2.clip(lower=-0.05), color=col, zorder=3)
ax.set_yticks(np.arange(len(order))); ax.set_yticklabels(order["feature set"], fontsize=8.5)
ax.axvline(order.leave_site_R2.max(), color="#888", ls="--", lw=0.8)
ax.set_xlabel("leave-site-out $R^2$"); ax.set_xlim(-0.1, 0.8)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Fewer inputs, same skill — feature-group ablation", fontsize=10, fontweight="bold")
plt.show()

**Takeaway:** meteorology alone already matches the full 14-feature model; greenness
alone is noise (R² ≈ 0); the only satellite signal that helps is **water + LST**. In fact the
**top-7 inputs by importance — 6 meteorology + NDWI — reach leave-site R² ≈ 0.72, matching the
full model with half the features** (top-6 ≈ 0.72 too). So the model can be trimmed hard with
no loss, which the VIF and AIC/BIC selection below make formal.

## 9. Feature selection I — multicollinearity (VIF)

Several predictors are near-duplicates: the greenness indices are all monotone functions
of NIR-Red, and ETo is a combination of the met variables. We quantify this with the
variance inflation factor (VIF) and prune the redundant features.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

def vif(cols):
    Z = StandardScaler().fit_transform(d[cols].values)
    return pd.Series({f: 1/max(1e-9, 1-LinearRegression().fit(
        np.delete(Z, j, 1), Z[:, j]).score(np.delete(Z, j, 1), Z[:, j]))
        for j, f in enumerate(cols)}).sort_values(ascending=False)

print("VIF (full 14 features):"); print(vif(FEATS).round(1).to_string())

cmat = d[FEATS].corr().abs()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cmat.values, cmap="RdBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(FEATS))); ax.set_xticklabels(FEATS, rotation=90, fontsize=6.5)
ax.set_yticks(range(len(FEATS))); ax.set_yticklabels(FEATS, fontsize=6.5)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
ax.set_title("|correlation| among predictors", fontsize=10, fontweight="bold")
plt.show()

# iterative elimination: drop the highest VIF until all <= 10
cols = list(FEATS)
while len(cols) > 2 and vif(cols).max() > 10:
    worst = vif(cols).idxmax(); print("drop", worst, " VIF", round(vif(cols).max(), 1)); cols.remove(worst)
SELECTED = cols
print("\nVIF-pruned set (", len(SELECTED), "):", SELECTED)

VIF flags the greenness indices as severe (EVI2/SAVI in the thousands). Eliminating
the worst leaves an ~11-feature decorrelated set (all VIF ≤ ~8).

## 10. Feature selection II — AIC/BIC (linear model)

AIC/BIC are defined for the maximum-likelihood (OLS) model, so we use them for forward
stepwise selection of a linear ET model. They are **not** valid for the tree ensembles (no
likelihood, no well-defined parameter count), so model choice among those stays on
leave-site CV.

In [ ]:
import statsmodels.api as sm
y = d.ET_closed_mm.values
Xs = pd.DataFrame(StandardScaler().fit_transform(d[FEATS].values), columns=FEATS, index=d.index)

def ic(cols):
    m = sm.OLS(y, sm.add_constant(Xs[cols])).fit(); return m.aic, m.bic

def forward(which):
    rem, sel, cur = list(FEATS), [], 1e18
    while rem:
        f, s = min(((f, ic(sel + [f])[0 if which == "aic" else 1]) for f in rem), key=lambda t: t[1])
        if s < cur - 1e-6:
            cur = s; sel.append(f); rem.remove(f)
        else:
            break
    return sel

aic_set, bic_set = forward("aic"), forward("bic")
for nm, cs in [("full(14)", FEATS), ("VIF", SELECTED), ("AIC-step", aic_set), ("BIC-step", bic_set)]:
    a, b = ic(cs); print(f"  {nm:<9} k={len(cs):<3} AIC={a:8.1f}  BIC={b:8.1f}")
print("\nBIC-selected:", bic_set)

BIC (the stricter penalty) drops **every greenness index**, keeping only water
(NDWI) and thermal (LST) among the satellite features — the same verdict as VIF and the
feature-importance ablation.

## 11. Retrain all models on the selected features → pick + save the best

We retrain the whole model zoo on the pruned features and rank by leave-site R² (spatial
transfer). The winner is refit on all the data and saved as the production model.

In [ ]:
import joblib, json
from sklearn.base import clone

rows = []
for name, mk in ZOO.items():
    rows.append((name, round(evaluate(mk, d, "site", feats=SELECTED)[0], 3)))
rank = pd.DataFrame(rows, columns=["model", "leave_site_R2"]).sort_values("leave_site_R2", ascending=False)
print(rank.to_string(index=False))

best = rank.iloc[0]["model"]
prod = clone(ZOO[best]()).fit(d[SELECTED].values, d.ET_closed_mm.values)
meta = {"model": best, "features": SELECTED, "leave_site_R2": float(rank.iloc[0]["leave_site_R2"]),
        "n_train": int(len(d)), "n_sites": int(d.SITE_ID.nunique())}
joblib.dump({**meta, "model": prod}, f"{PROC}/final_model.joblib")  # fitted model wins the key
json.dump(meta, open(f"{PROC}/final_model.json", "w"), indent=2)
print(f"\nPRODUCTION MODEL: {best} on {len(SELECTED)} features  ->  saved final_model.joblib")

**Result:** ExtraTrees on the decorrelated feature set is the production model
(leave-site R² ≈ 0.72). Multicollinearity (VIF), AIC/BIC, and the importance ablation all
converge on the same parsimonious set — **water + thermal + meteorology, greenness dropped**
— and this is the model `04_spatial_prediction` uses to map ET.

---
# Part 3 — Spatial-upscaling validation

In [ ]:
# The 14 predictors: 7 satellite + 7 meteorology. Target is measured closed ET.
SAT = ["LAI", "EVI2", "SAVI", "NDVI", "NDWI", "MNDWI", "LST_K"]
MET = ["TA_ERA", "VPD_ERA", "SW_IN_ERA", "WS_ERA", "ETo_mm", "DOY_sin", "DOY_cos"]
FEATS = SAT + MET
EVERGLADES = ["US-Esm", "US-TaS", "US-Skr", "US-Elm", "US-EvM"]

d = pd.read_parquet(f"{PROC}/more_sites_table.parquet")
SITES = sorted(d.SITE_ID.unique())
print(f"{len(d)} overpass matches | {len(SITES)} sites | {len(FEATS)} features")
print("target: ET_closed_mm (measured, closure-corrected daily ET)")
d[["SITE_ID", "year"] + FEATS + ["ET_closed_mm"]].head()

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error

def evaluate(make_model, data, scheme, feats=FEATS):
    """Return (R2, MAE, y_true, y_pred) for one CV scheme.
    kfold=predict at monitored sites; year=predict unseen years;
    site=predict a completely unseen tower (spatial upscaling)."""
    yt, yp = [], []
    if scheme == "kfold":
        for tri, tei in KFold(10, shuffle=True, random_state=0).split(data):
            m = clone(make_model()).fit(data.iloc[tri][feats].values, data.iloc[tri].ET_closed_mm.values)
            yp.append(m.predict(data.iloc[tei][feats].values)); yt.append(data.iloc[tei].ET_closed_mm.values)
    elif scheme == "year":
        for s in sorted(data.SITE_ID.unique()):
            ds = data[data.SITE_ID == s]
            for y in sorted(ds.year.dropna().unique()):
                tr, te = ds[ds.year != y], ds[ds.year == y]
                if len(te) < 5 or len(tr) < 15: continue
                m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
                yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    else:  # leave-site-out
        for s in sorted(data.SITE_ID.unique()):
            tr, te = data[data.SITE_ID != s], data[data.SITE_ID == s]
            if len(te) < 5: continue
            m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
            yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    return r2_score(yt, yp), mean_absolute_error(yt, yp), yt, yp

print("CV evaluator ready (schemes: kfold, year, site)")

## 5 Everglades sites vs 13 diverse wetlands

We run leave-site-out (predict a fully unseen tower) on the 5-site Everglades subset and
on the full 13-site network, with our best upscaler (ExtraTrees) and the Gaussian process.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODELS = {
    "ExtraTrees": lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
    "GaussProc":  lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
                       kernel=ConstantKernel() * RBF() + WhiteKernel(), normalize_y=True, alpha=1e-3, random_state=0)),
}
d5 = d[d.SITE_ID.isin(EVERGLADES)]
res = {}
for mn, mk in MODELS.items():
    for label, data in [("5 Everglades", d5), ("13 wetlands", d)]:
        rk = evaluate(mk, data, "kfold")[0]; ry = evaluate(mk, data, "year")[0]; rs = evaluate(mk, data, "site")[0]
        res[(mn, label)] = (rk, ry, rs)
        print(f"  {mn:<11}{label:<14} kfold={rk:5.2f}  leave-year={ry:5.2f}  leave-site={rs:5.2f}", flush=True)

## The picture: harder tests need more sites

In [ ]:
schemes = ["K-fold\n(monitored)", "Leave-year\n(unseen yr)", "Leave-site\n(unseen tower)"]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True, constrained_layout=True)
x = np.arange(3); w = 0.36
for ax, mn in zip(axes, ["ExtraTrees", "GaussProc"]):
    v5, v13 = res[(mn, "5 Everglades")], res[(mn, "13 wetlands")]
    ax.bar(x - w/2, np.clip(v5, -1.05, 1), w, color="#B0C4DE", label="5 Everglades", zorder=3)
    ax.bar(x + w/2, np.clip(v13, -1.05, 1), w, color="#2C5F8A", label="13 wetlands", zorder=3)
    ax.axhline(0, color="#8a8a8a", lw=0.8)
    for xi, (a, b) in enumerate(zip(v5, v13)):
        ax.text(xi - w/2, max(a, 0)+0.03, f"{a:.2f}", ha="center", fontsize=6, color=INK)
        ax.text(xi + w/2, max(b, 0)+0.03, f"{b:.2f}", ha="center", fontsize=6, color=INK)
    ax.set_title(mn, fontsize=9); ax.set_xticks(x); ax.set_xticklabels(schemes, fontsize=6.8)
    ax.set_ylim(-1.15, 1.0)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
axes[0].set_ylabel("$R^2$"); axes[1].legend(frameon=False, fontsize=7, loc="lower right")
fig.suptitle("Upscaling to an unseen tower needs training-set diversity",
             fontsize=10, fontweight="bold")
fig.savefig(f"{FIG}/cv_comparison_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

## Predicted vs observed at held-out towers (13 sites)

Every point is an overpass at a tower the model never saw in training.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
_, _, yt, yp = evaluate(lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1), d, "site")
from sklearn.metrics import r2_score, mean_absolute_error
fig, ax = plt.subplots(figsize=(4.0, 4.0), constrained_layout=True)
ax.scatter(yt, yp, s=9, alpha=0.35, color="#2C5F8A", edgecolor="none")
lim = [0, max(yt.max(), yp.max())*1.05]
ax.plot(lim, lim, "--", color="#8a8a8a", lw=0.9)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Observed ET (mm/day)"); ax.set_ylabel("Predicted ET (mm/day)")
ax.text(0.05, 0.92, f"$R^2$={r2_score(yt,yp):.2f}\nMAE={mean_absolute_error(yt,yp):.2f} mm/d",
        transform=ax.transAxes, fontsize=8, va="top")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Leave-site-out prediction, 13 wetlands", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/upscaling_scatter_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

**The finding, reproduced live:** the bottleneck to satellite ET upscaling in coastal
wetlands is **training-set diversity**, not the model or the features. Five spectrally
near-identical Everglades marshes can't teach a transferable relationship; thirteen
wetlands spanning different climates, salinities and canopies can (leave-site $R^2\approx0.7$).

---
# Part 4 — Mapping ET at 30 m over the reserves

We apply the validated model across seven National Estuarine Research Reserve boundaries
(bundled in `shp_predict/`), predicting daily ET at **30 m**, masking open water, and
clipping to each reserve. The 30 m maps are **pre-computed and included** in
`data/processed/reserve_maps/`, so this section **loads** them and runs in seconds.

**This default path needs no external service** — it uses only the bundled shapefiles and the
saved maps, so it keeps working even if Planetary Computer or gridMET is ever unavailable.
Set **`RECOMPUTE = True`** below to regenerate from scratch instead — for each reserve that
re-downloads a clear Landsat scene + gridMET, recomputes the seven indices per pixel, runs
the ExtraTrees model, masks water, and clips to the boundary (needs internet, ~10 min). The
whole per-reserve pipeline lives in the portable `reserve_et.py` module.

In [ ]:
RECOMPUTE = False    # True -> re-download imagery + re-predict live (internet, ~10 min)
OUT = f"{PROC}/reserve_maps"
results = []
if RECOMPUTE:
    model, feats = RE.load_production_model()
    cat, das = RE.open_catalog(), RE.open_gridmet()
    for shp in sorted(glob.glob(f"{RE.SHP_DIR}/*/*.shp")):
        name = os.path.basename(os.path.dirname(shp))
        r = RE.predict_reserve(shp, model, cat, das, mask_water=True, feats=feats)  # downloads + predicts
        if r is None:
            print(f"  {name}: no clear scene"); continue
        results.append(r); print(f"  {name}: {r['date']}  mean ET {r['mean_ET']} mm/day")
else:
    summ = pd.read_csv(f"{OUT}/reserve_ET_summary.csv").set_index("reserve")
    for f in sorted(glob.glob(f"{OUT}/*.npz")):
        z = np.load(f, allow_pickle=True); name = os.path.basename(f).split("_")[0]
        epsg = int(z["epsg"])
        poly = gpd.read_file(glob.glob(f"{RE.SHP_DIR}/{name}/*.shp")[0]).to_crs(epsg).union_all()
        s = summ.loc[name]
        results.append(dict(reserve=name, et=z["et"], x=z["x"], y=z["y"], epsg=epsg, poly=poly,
            date=str(z["date"]), cloud=float(s.cloud), mean_ET=float(s.mean_ET),
            min_ET=float(s.min_ET), max_ET=float(s.max_ET), pixels=int(s.pixels),
            water_px=int(s.water_px), inside_px=int(s.inside_px)))
        print(f"  loaded {name}: {z['date']}  mean ET {s.mean_ET} mm/day")
print(f"\n{len(results)} reserves ready ({'recomputed live' if RECOMPUTE else 'loaded from included maps'})")

## 4.1 Summary table

In [ ]:
summary = pd.DataFrame([{k: r[k] for k in
    ["reserve","date","cloud","mean_ET","min_ET","max_ET","pixels","water_px","inside_px","epsg"]}
    for r in results])
summary["water_pct"] = (100 * summary.water_px / summary.inside_px).round(0)
summary

## 4.2 The maps — one clipped 30 m ET map per reserve, shared colour scale

In [ ]:
PAL = ["#DEC29B","#EDD9A6","#FFF4AD","#C3E683","#6BCC5C","#3BB369","#20998F","#16678A","#114982"]
cmap = LinearSegmentedColormap.from_list("et", PAL, N=256); cmap.set_bad("#eeeeea")
n = len(results); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3.4, nrow*3.2))
axes = np.atleast_1d(axes).ravel()
for ax in axes: ax.axis("off")
for ax, r in zip(axes, results):
    ext = [r["x"].min(), r["x"].max(), r["y"].min(), r["y"].max()]
    im = ax.imshow(np.ma.masked_invalid(r["et"]), origin="upper", extent=ext, cmap=cmap, vmin=1, vmax=6)
    gpd.GeoSeries([r["poly"]], crs=r["epsg"]).boundary.plot(ax=ax, color="#333", lw=0.6)
    ax.set_title(f"{r['reserve']}  {r['date']}\nmean {r['mean_ET']} mm/d", fontsize=8.5, fontweight="bold")
    ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
cb = fig.colorbar(im, ax=axes.tolist(), fraction=0.02, pad=0.02, extend="both")
cb.set_label("predicted daily ET (mm/day)")
fig.suptitle("Predicted ET across NERR coastal-wetland reserves (open water masked)",
             fontsize=12, fontweight="bold")
plt.show()

## 4.3 Zoom on one reserve

In [ ]:
pick = "WKB"      # any reserve code in the summary
r = next(x for x in results if x["reserve"] == pick)
fig, ax = plt.subplots(figsize=(7, 6))
ext = [r["x"].min(), r["x"].max(), r["y"].min(), r["y"].max()]
im = ax.imshow(np.ma.masked_invalid(r["et"]), origin="upper", extent=ext, cmap=cmap, vmin=1, vmax=6)
gpd.GeoSeries([r["poly"]], crs=r["epsg"]).boundary.plot(ax=ax, color="#333", lw=0.8)
cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02, extend="both")
cb.set_label("predicted daily ET (mm/day)"); cb.outline.set_visible(False)
ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title(f"{r['reserve']} — predicted ET, {r['date']} (open water masked)", fontsize=12, fontweight="bold")
plt.show()

## 4.4 (Optional) export GeoTIFFs
The 30 m GeoTIFFs are already in `data/processed/reserve_maps/` (and in the published
dataset). If you re-ran with `RECOMPUTE = True`, this writes the fresh rasters.

In [ ]:
if RECOMPUTE:
    os.makedirs(OUT, exist_ok=True)
    for r in results: RE.save_outputs(r, OUT)
    print(f"wrote {len(results)} GeoTIFFs + npz to {OUT}")
else:
    print("GeoTIFFs already present in", OUT)

## 4.5 (Optional) Predict over *your own* area

Bring any polygon shapefile — upload `your_area.shp` (with its `.shx/.dbf/.prj` sidecars)
into this environment, set the path below, and run. This predicts **live** for a new area,
so it needs internet and a working Planetary Computer + gridMET (gridMET covers the **U.S.**).
The result is drawn here and written as a GeoTIFF + `.npz` to `reserve_maps/`.

In [ ]:
MY_SHAPEFILE = ""     # e.g. "my_area/my_wetland.shp"   (leave "" to skip)
if not MY_SHAPEFILE:
    print("Set MY_SHAPEFILE to a shapefile path to predict ET over your own polygon.")
else:
    model, feats = RE.load_production_model()
    cat, das = RE.open_catalog(), RE.open_gridmet()
    r = RE.predict_reserve(MY_SHAPEFILE, model, cat, das, mask_water=True, feats=feats)
    if r is None:
        print("No clear scene found for that area / time window — try a larger area or another season.")
    else:
        print(f"{r['reserve']}: {r['date']}  mean ET {r['mean_ET']} mm/day  ({r['pixels']:,} land px)")
        fig, ax = plt.subplots(figsize=(7, 6))
        ext = [r["x"].min(), r["x"].max(), r["y"].min(), r["y"].max()]
        im = ax.imshow(np.ma.masked_invalid(r["et"]), origin="upper", extent=ext, cmap=cmap, vmin=1, vmax=6)
        gpd.GeoSeries([r["poly"]], crs=r["epsg"]).boundary.plot(ax=ax, color="#333", lw=0.8)
        cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02, extend="both")
        cb.set_label("predicted daily ET (mm/day)"); cb.outline.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_title(f"Predicted ET over your area — {r['date']} (open water masked)", fontsize=12, fontweight="bold")
        RE.save_outputs(r, OUT); print("saved GeoTIFF + npz to", OUT)
        plt.show()

**Caveats.** Resolution is 30 m from the optical indices (thermal LST ~100 m resampled);
meteorology is one gridMET value per reserve/date. Open water is masked; marsh with standing
water is kept. Each reserve uses its own clearest scene, so **dates differ** — fix the month
for a strict cross-reserve comparison. These are **new locations**: leave-site R²≈0.72 was
measured on the 13 training wetlands, so read absolute values as indicative and the spatial
pattern as the product.

---
# Conclusions

- **Meteorology carries the signal.** gridMET reference ET plus a few weather terms explain
  most of the daily variance; satellite indices mainly add spatial texture. Four independent
  feature-selection methods (permutation importance, group ablation, VIF, AIC/BIC) agree that
  **~6–7 inputs** match the full fourteen.
- **Honest spatial skill:** under **leave-one-site-out** cross-validation the production
  model (ExtraTrees on 11 VIF-pruned features) reaches **R² ≈ 0.72** — i.e. genuine transfer
  to *unseen* wetlands, not just interpolation within a site.
- **Diversity is the bottleneck.** Five spectrally similar Everglades sites alone fail to
  generalise; a diverse 13-site network succeeds. More *varied* towers — not a fancier model
  — is what would improve the maps.
- **Product:** 30 m ET maps over seven NERR reserves (open water masked), exported as
  GeoTIFFs for GIS. Treat absolute values as indicative at these new locations; the spatial
  pattern is the deliverable.

## License & citations
Derived table, model, and maps are released **CC-BY-4.0**. Underlying data keep their own
terms (AmeriFlux/FLUXNET CC-BY-4.0; Landsat USGS public domain; gridMET public domain; ERA5
Copernicus C3S). See `docs/CITATIONS.md`. Funded by **NSF OAC-2118329** (I-GUIDE Institute).